# 📈 MLflow Integration Example

Track `mlprep` pipeline executions using MLflow.

In [ ]:
!pip install -q "mlprep-rust==0.3.1" pandas pyarrow mlflow

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd() / 'outputs'
BASE.mkdir(exist_ok=True)

np.random.seed(42)
df = pd.DataFrame(np.random.rand(10, 2), columns=['col1', 'col2'])
df.to_csv('raw_data.csv', index=False)
print('Generated raw_data.csv')
df

In [ ]:
pipeline_yaml = '''name: mlflow_experiment
inputs:
  - path: raw_data.csv
    format: csv
steps:
  - type: select
    columns: [col1, col2]
outputs:
  - path: outputs/output.parquet
    format: parquet
'''

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml)
print(pipeline_yaml)

In [ ]:
import mlflow
import subprocess
import os

mlflow.set_experiment('mlprep_experiment')
with mlflow.start_run() as run:
    print(f'Run ID: {run.info.run_id}')
    mlflow.log_artifact('pipeline.yaml')
    try:
        subprocess.run(['mlprep', 'run', 'pipeline.yaml', '--streaming', '--memory-limit', '1GB'], check=True)
        mlflow.log_param('status', 'SUCCESS')
        if os.path.exists('outputs/output.parquet'):
            mlflow.log_artifact('outputs/output.parquet')
            df = pd.read_parquet('outputs/output.parquet')
            mlflow.log_metric('rows', len(df))
            print(f'Logged {len(df)} rows')
    except Exception as e:
        mlflow.log_param('status', 'FAILED')
        print(e)

In [ ]:
experiment = mlflow.get_experiment_by_name('mlprep_experiment')
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print(runs[['run_id', 'params.status', 'metrics.rows']].head())